In [30]:
import os
from openai import OpenAI
from mem0 import MemoryClient  
from dotenv import load_dotenv
import hashlib
import os
import tempfile
from pathlib import Path
from datetime import datetime

import streamlit as st
from langchain_core.messages import HumanMessage
from src.agentic_workflow.agent import graph
load_dotenv()

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
MEM0_API_KEY = os.environ["MEM0_API_KEY"]

llm_client = OpenAI(api_key=OPENAI_API_KEY)
mem_client = MemoryClient(api_key=MEM0_API_KEY)

2026-09-27 10:54:37,901 - INFO - HTTP Request: GET https://api.mem0.ai/v1/ping/ "HTTP/1.1 200 OK"


In [32]:
AGENT_ID = "support-agent-v1"

def retrieve_memories(user_id: str, query: str, limit: int = 5):
    """Fetch relevant long-term memories from Mem0."""
    results = mem_client.search(
        user_id=user_id,
        agent_id=AGENT_ID,
        query=query,
        limit=limit,
    )
    return results  # list of dicts with 'text', 'score', 'metadata', etc.

In [33]:
from src.agentic_workflow.agent import graph

config = {
        "configurable": {"thread_id": "default_thread"}
    }

In [43]:
class memory:
    def __init__(self,user_prompt:str,user_id:str,agent_id:str,limit:int=5):

        self.user_prompt = user_prompt
        self.user_id = user_id
        self.limit = limit
        self.agent_id = agent_id

    def retrive(self):
        results = mem_client.search(
                user_id=self.user_id,
                agent_id=self.agent_id,
                query=self.user_prompt,
                limit=self.limit,
            )
        return results
    def store_memory(self):
        """Store a new memory in Mem0."""
        mem_client.add(
            user_id=self.user_id,
            agent_id=self.agent_id,
            text=self.user_prompt,
        )
    def build_prompt(self,memories: list[dict]) -> str:
        memory_block = ""
        if memories:
            context = "\n".join(f"- {m['text']}" for m in memories)
            memory_block = f"Known user context:\n{context}\n\n"

        system = (
            "You are a helpful support agent. Use the known user context when relevant, "
            "and avoid asking for information already present there."
        )
        return f"{system}\n\n{memory_block}User: {self.user_prompt}\nAssistant:"

    def semantic_memory_for_facts(self) -> str | None:
        """Heuristic: store preferences or explicit facts."""
        triggers = ["I prefer", "I usually", "My name is", "I am working on", "I live in"]
        lower_msg = self.user_prompt.lower()
        if any(t.lower() in lower_msg for t in triggers):
            return self.user_prompt
        # optionally parse structured memory using another LLM call
        return None
    

In [35]:
awareness=memory(user_id="ilias", agent_id="agent1")

In [ ]:
prompt="how are you?"
result = graph.invoke({"messages": [HumanMessage(content=prompt)]},config=config)
print(result["messages"][-1].content)

2026-09-27 10:55:45,196 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


hi


2026-09-27 10:56:17,317 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


I'm here and ready to assist you! If you have any questions or need help, feel free to ask. How can I help you today?


In [ ]:
from src.agentic_workflow import agent


def call_llm(prompt: str) -> str:
        result = agent.invoke({"messages": [HumanMessage(content=prompt)]},config=config)
    return result

In [22]:
def store_memory(user_id: str, text: str):
    """Store a new memory in Mem0."""
    mem_client.add(
        user_id=user_id,
        agent_id=AGENT_ID,
        text=text,
    )

In [ ]:
def build_prompt(user_message: str, memories: list[dict]) -> str:
    memory_block = ""
    if memories:
        context = "\n".join(f"- {m['text']}" for m in memories)
        memory_block = f"Known user context:\n{context}\n\n"

    system = (
        "You are a helpful support agent. Use the known user context when relevant, "
        "and avoid asking for information already present there."
    )
    return f"{system}\n\n{memory_block}User: {user_message}\nAssistant:"

In [ ]:
def call_llm(prompt: str) -> str:
    response = llm_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return response.choices[0].message.content.strip()

In [ ]:
def semantic_memory_for_facts(user_message: str, assistant_reply: str) -> str | None:
    """Heuristic: store preferences or explicit facts."""
    triggers = ["I prefer", "I usually", "My name is", "I am working on", "I live in"]
    lower_msg = user_message.lower()
    if any(t.lower() in lower_msg for t in triggers):
        return user_message
    # optionally parse structured memory using another LLM call
    return None

In [ ]:
def chat_with_memory(user_id: str):
    print("Start chatting. Type 'exit' to quit.")
    while True:
        user_message = input("You: ")
        if user_message.lower() in {"exit", "quit"}:
            break

        # 1) Retrieve relevant memories
        memories = retrieve_memories(user_id=user_id, query=user_message)

        # 2) Build prompt with memory
        prompt = build_prompt(user_message, memories)

        # 3) Call LLM
        assistant_reply = call_llm(prompt)
        print(f"Agent: {assistant_reply}\n")

        # 4) Store new memory when appropriate
        candidate = semantic_memory_for_facts(user_message, assistant_reply)
        if candidate:
            store_memory(
                user_id=user_id,
                text=candidate,
            )

In [64]:
class memory:
    def __init__(self,user_prompt:str,user_id:str,agent_id:str,limit:int=5):
        self.user_prompt = user_prompt
        self.user_id = user_id
        self.limit = limit
        self.agent_id = agent_id

    def retrive(self):
        results = mem_client.search(
                filters={'user_id': self.user_id, 'agent_id': self.agent_id},
                query=self.user_prompt,
                limit=self.limit,
            )
        return results
    def store_memory(self):
        """Store a new memory in Mem0."""
        mem_client.add(
            user_id=self.user_id,
            agent_id=self.agent_id,
            messages=self.user_prompt,
        )
    def build_prompt(self,memories: list[dict]) -> str:
        memory_block = ""
        if memories:
            context = "\n".join(f"- {m}" for m in memories)
            memory_block = f"Known user context:\n{context}\n\n"

        system = (
            "You are a helpful support agent. Use the known user context when relevant, "
            "and avoid asking for information already present there."
        )
        return f"{system}\n\n{memory_block}User: {self.user_prompt}\nAssistant:"

    def semantic_memory_for_facts(self) -> str | None:
        """Heuristic: store preferences or explicit facts."""
        triggers = ["I prefer", "I usually", "My name is", "I am working on", "I live in"]
        lower_msg = self.user_prompt.lower()
        if any(t.lower() in lower_msg for t in triggers):
            return self.user_prompt
        # optionally parse structured memory using another LLM call
        return None
    

In [69]:
prompt="what is my name?"

In [70]:
m=memory(user_prompt=prompt, user_id="ilias", agent_id="agent1")

In [71]:
long_tm=m.retrive()
build_prompt=m.build_prompt(long_tm)
agent_response = graph.invoke({"messages": [HumanMessage(content=build_prompt)]},config=config)
x=m.semantic_memory_for_facts()
print(agent_response["messages"][-1].content)
if x:
    m.store_memory()

2026-09-27 11:30:25,515 - INFO - HTTP Request: POST https://api.mem0.ai/v3/memories/search/ "HTTP/1.1 200 OK"
2026-09-27 11:30:27,198 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Your name is Ilias. How can I assist you today?
